In [1]:
#1 PydanticOutputParser
%pip --version

pip 25.0.1 from d:\hanwha_0902\hanwha_0902\ex0916\.venv\Lib\site-packages\pip (python 3.12)

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install -qU langchain_teddynote

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test_0916")

llm=ChatOpenAI(temperature=0.5, model_name="gpt-4o")

email_conversation = """From: 안철수(chulsoo.ahn@powerofpeople.com)
To: 이재명 (jaemyung.lee@togetherpeople.com)
Subject: "6.25" 전쟁 협의문 관련 미팅 일정 제안

안녕하세요, 이재명 대통령님,

저는 국민의 힘의 안철수 의원입니다. 

금주 금요일 (9월 18일) 오전 11시에 미팅을 제안합니다.

청와대에서 만나 이야기를 나눌 수 있을까요?

안철수
대표의원
국민의 힘
"""

from itertools import chain
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    "다음의 이메일 내용 중 중요한 내용을 추출해 주세요.\n\n{email_conversation}"
)

llm = ChatOpenAI(temperature=0.6, model_name="gpt-4o")

chain = prompt | llm

answer = chain.stream({"email_conversation": email_conversation})

output = stream_response(answer, return_output=True)

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
- 발신자: 안철수 의원 (국민의 힘)
- 수신자: 이재명 대통령
- 주제: "6.25" 전쟁 협의문 관련 미팅
- 제안된 일정: 9월 18일 금요일 오전 11시
- 장소: 청와대

In [12]:
from langchain_teddynote.messages import stream_response
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from langchain_teddynote import logging
from dotenv import load_dotenv

load_dotenv()
logging.langsmith("test_0916")

llm=ChatOpenAI(temperature=0.5, model_name="gpt-4o")

email_conversation = """From: 안철수(chulsoo.ahn@powerofpeople.com)
To: 이재명 (jaemyung.lee@togetherpeople.com)
Subject: "6.25" 전쟁 협의문 관련 미팅 일정 제안

안녕하세요, 이재명 대통령님,

저는 국민의 힘의 안철수 의원입니다. 

금주 금요일 (9월 18일) 오전 11시에 미팅을 제안합니다.

청와대에서 만나 이야기를 나눌 수 있을까요?

안철수
대표의원
국민의 힘
"""

class EmailSummary(BaseModel):
    person: str= Field(description="메일을 보낸 사람")
    email: str = Field(description= "메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")

parser = PydanticOutputParser(pydantic_object=EmailSummary)

print(parser.get_format_instructions())

prompt = PromptTemplate.from_template("""
    너는 AI 어시스턴트이다. 아래 내용에 대해 한국말로 답하도록.kwargs=

    QUESTION: {question}

    EMAIL_CONVERSATION: {email_conversation}

    FORMAT: {format}
    """
)

prompt = prompt.partial(format=parser.get_format_instructions())
prompt

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required": ["person", "email", "subject", "summary", "date"]}
```


PromptTemplate(input_variables=['email_conversation', 'question'], input_types={}, partial_variables={'format': 'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"person": {"description": "메일을 보낸 사람", "title": "Person", "type": "string"}, "email": {"description": "메일을 보낸 사람의 이메일 주소", "title": "Email", "type": "string"}, "subject": {"description": "메일 제목", "title": "Subject", "type": "string"}, "summary": {"description": "메일 본문을 요약한 텍스트", "title": "Summary", "type": "string"}, "date": {"description": "메일 본문에 언급된 미팅 날짜와 시간", "title": "Date", "type": "string"}}, "required

In [13]:
chain = prompt | llm

response = chain.stream(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해 주세요.",
    }
)

output = stream_response(response, return_output=True)

```json
{
    "person": "안철수",
    "email": "chulsoo.ahn@powerofpeople.com",
    "subject": "\"6.25\" 전쟁 협의문 관련 미팅 일정 제안",
    "summary": "안철수 의원이 이재명 대통령에게 9월 18일 금요일 오전 11시에 청와대에서 미팅을 제안합니다.",
    "date": "9월 18일 금요일 오전 11시"
}
```

In [14]:
structured_output = parser.parse(output)
print(structured_output)

person='안철수' email='chulsoo.ahn@powerofpeople.com' subject='"6.25" 전쟁 협의문 관련 미팅 일정 제안' summary='안철수 의원이 이재명 대통령에게 9월 18일 금요일 오전 11시에 청와대에서 미팅을 제안합니다.' date='9월 18일 금요일 오전 11시'


In [15]:
structured_output.email

'chulsoo.ahn@powerofpeople.com'

In [17]:
chain = prompt | llm | parser
response= chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용 중 주요 내용을 추출해주세요.",

    }
)

response

EmailSummary(person='안철수', email='chulsoo.ahn@powerofpeople.com', subject='"6.25" 전쟁 협의문 관련 미팅 일정 제안', summary='안철수 의원이 이재명 대통령에게 6.25 전쟁 협의문 관련 미팅을 제안하며, 금주 금요일 오전 11시에 청와대에서 만나기를 요청하고 있습니다.', date='9월 18일 오전 11시')

In [18]:
#2 with_structured_output()바인딩

llm = ChatOpenAI(
    temperature = 0.5, model_name="gpt-4o"
)

llm.invoke("대한민국 최고의 라멘 맛집은 어디야?")

AIMessage(content='대한민국에는 많은 훌륭한 라멘 맛집이 있지만, 몇 가지 주목할 만한 곳을 소개해드릴게요.\n\n1. **하코네라멘** - 서울에 위치한 이곳은 진한 육수와 쫄깃한 면발로 유명합니다. 특히 돈코츠 라멘이 인기입니다.\n\n2. **멘야산다이메** - 서울과 부산에 지점이 있는 이곳은 일본 현지의 맛을 재현하여 많은 라멘 애호가들에게 사랑받고 있습니다.\n\n3. **이치란 라멘** - 일본에서 유명한 이치란 라멘은 서울에 지점을 두고 있으며, 개인 맞춤형 라멘을 즐길 수 있어 인기가 많습니다.\n\n4. **라멘모토** - 서울 홍대에 위치한 이곳은 다양한 종류의 라멘과 함께 사이드 메뉴도 훌륭합니다.\n\n5. **우마이도** - 대구에 위치한 이곳은 정통 일본 라멘을 맛볼 수 있는 곳으로, 특히 쇼유 라멘이 유명합니다.\n\n각 지역마다 특색 있는 라멘 맛집이 많으니, 방문하시는 곳에서 가까운 맛집을 찾아보시는 것도 좋을 것 같습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 271, 'prompt_tokens': 18, 'total_tokens': 289, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'sy

In [19]:
llm_with_structured = ChatOpenAI(
    temperature=0.5, model_name="gpt-4o"
).with_structured_output(EmailSummary)

answer = llm_with_structured.invoke(email_conversation)
answer.person

'안철수'

In [3]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_teddynote import logging

load_dotenv()
logging.langsmith("test_0916")

output_parser = CommaSeparatedListOutputParser()
format_instructions = output_parser.get_format_instructions()

print(format_instructions)

LangSmith 추적을 시작합니다.
[프로젝트명]
test_0916
Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`


In [4]:
prompt = PromptTemplate(
    template="{subject} 5가지를 한국어로 답변해줘.\n{format_instructions}",
    input_variables=["subject"],
    partial_variables={"format_instructions": format_instructions},
)

model = ChatOpenAI(temperature=0.5)

chain = prompt | model | output_parser

answer = chain.invoke({"subject": "대한민국 최고의 라멘 식당"})
answer

['이치란', '신라면세트', '고라멘', '라멘 오카', '라멘 노바']

In [5]:
for superdupa in chain.stream({"subject": "대한민국 맛집"}):
    print(superdupa)

['고기구이']
['불고기']
['국수']
['떡볶이']
['삼겹살']
